# Retrieval Evaluation

This notebook evaluates one checkpoint through the same code path used by the CLI.
It works for EEG, MEG, or fMRI, and it can also run shared-only retrieval when you provide a shared manifest.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


            from pathlib import Path
            import pandas as pd
            import torch

            from src.checkpoints import resolve_existing_checkpoint_path
            from src.eval_utils import (
                build_model,
                create_dataloader,
                load_checkpoint,
                load_clip_cache,
                load_config,
            )
            from src.evaluate import evaluate

            CONFIG_PATH = ROOT / "config.yaml"
            MODALITY = "meg"
            SUBJECT = 1
            SPLIT = "test"
            SHARED_ONLY = False
            SHARED_MANIFEST = None  # e.g. ROOT / "data/manifests/conversion_pools/eeg_fmri.txt"

            config = load_config(CONFIG_PATH)
            if SHARED_MANIFEST is not None:
                config.setdefault("data", {})["shared_manifest_path"] = str(SHARED_MANIFEST)
            elif SHARED_ONLY:
                raise ValueError("Set SHARED_MANIFEST when SHARED_ONLY=True")

            checkpoint_path = resolve_existing_checkpoint_path(
                MODALITY,
                SUBJECT,
                kind="best",
                shared_only=SHARED_ONLY,
                shared_manifest_path=str(SHARED_MANIFEST) if SHARED_MANIFEST else None,
            )
            checkpoint_path


In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
clip_dict = load_clip_cache(config)
loader = create_dataloader(
    config,
    MODALITY,
    SPLIT,
    subject=SUBJECT,
    shared_only=SHARED_ONLY,
    quiet=False,
    shuffle=False,
)
sample_x = loader.dataset[0]["x"]
model = build_model(config, MODALITY, sample_x, device)
load_checkpoint(model, checkpoint_path, device)
metrics = evaluate(model, loader, clip_dict, device)

pd.DataFrame(
    [
        {
            "modality": MODALITY,
            "subject": SUBJECT,
            "split": SPLIT,
            "shared_only": SHARED_ONLY,
            "candidate_images": metrics["candidate_count"],
            "modality_to_image_top1": metrics["modality_to_image"]["top1"],
            "modality_to_image_top5": metrics["modality_to_image"]["top5"],
            "modality_to_image_two_way": metrics["modality_to_image"]["two_way"],
            "image_to_modality_top1": metrics["image_to_modality"]["top1"],
            "image_to_modality_top5": metrics["image_to_modality"]["top5"],
            "image_to_modality_two_way": metrics["image_to_modality"]["two_way"],
        }
    ]
)
